In [1]:
import pandas as pd

In [2]:
date_url = "https://raw.githubusercontent.com/seanyboi/rugbydata/main/data/v3/competition/competition_registry.parquet"
competitions = pd.read_parquet(date_url, engine="pyarrow")

In [3]:
competitions

,competition_id,competition_name
0,d0f3c9d2,Top 14
1,14e5c252,Internationals
2,25c90071,U20 Six Nations
3,5aa5983c,Japan Rugby League One
4,690eb70a,Premiership Womens Rugby
5,91f4b76e,Pro D2
6,cacc1678,Super Rugby
7,ab1bbd78,United Rugby Championship
8,1b875d20,Premiership
9,f5b03bf0,Womens Internationals


In [4]:
date_url = "https://raw.githubusercontent.com/seanyboi/rugbydata/main/data/v3/match/match_registry.parquet"
matches = pd.read_parquet(date_url, engine="pyarrow").sort_values(by="date", ascending=True)

In [5]:
matches

,match_id,home_team,away_team,date
0,c80caaac,Pau,Toulon,20260201
447,15952718,New Zealand 7s Mens,South Africa 7s Mens,20260201
448,da8a4327,Australia 7s Mens,Spain 7s Mens,20260201
449,1b24265f,Australia 7s Womens,New Zealand 7s Womens,20260201
450,dd95092e,Canada 7s Womens,Usa 7s Womens,20260201
...,...,...,...,...
439,57525c38,Reds,Moana Pasifika,20260523
440,83ce0416,Hurricanes,Highlanders,20260523
441,f6eb07aa,Fijian Drua,Force,20260523
442,ba6710b9,Blackrams Tokyo,Tokyo Sungoliath,20260523


In [6]:
date_url = "https://raw.githubusercontent.com/seanyboi/rugbydata/main/data/v3/dates/20231021.parquet"
date = pd.read_parquet(date_url, engine="pyarrow")

In [7]:
date

,match_id,competition_id,home_team_id,home_team,away_team_id,away_team
0,80271b54,83d92007,f11abcf7,England,446f2243,South Africa
1,e097517c,2c3df351,895ae027,Northampton Saints,47d5fe17,Bristol Rugby
2,a05987a3,2c3df351,0026b492,Saracens,aa4f64f5,Bath Rugby
3,32fbd92e,2c3df351,1ebc0e7a,Leicester Tigers,d7d74d92,Sale Sharks
4,2c6e5d60,50e6935b,8ab9f1c9,Zebre,0aa2b3cc,Ulster
5,b00fc50e,50e6935b,6f5af01b,Connacht,5fa975a5,Ospreys
6,ac6e6a90,50e6935b,bb49fc47,Lions,f54be954,Stormers
7,79c6b10a,50e6935b,973ef5cc,Dragons,242b5b2e,Edinburgh
8,f9896230,50e6935b,79c81be4,Cardiff Blues,af0bb046,Benetton Treviso
9,d1c4208d,50e6935b,afe6fb82,Munster,1b2d3d61,Sharks


In [8]:
selected_match = "f9896230" # Cardiff vs. Benetton 

In [9]:
matches[matches["match_id"] == selected_match]

,match_id,home_team,away_team,date


In [10]:
player_manifest_url = "https://raw.githubusercontent.com/seanyboi/rugbydata/main/data/v3/player/player_registry.parquet"
players = pd.read_parquet(player_manifest_url, engine="pyarrow")

In [11]:
players

,player_id,player_name
0,2ce98072,Facundo Isa
1,eab99316,Jack Maddocks
2,91e6f0b2,Dan Robson
3,33ad54dc,Reece Hewat
4,c3958443,Melvyn Jaminet
...,...,...
5271,1243ca83,Alia Bitonci
5272,2f8f2779,Aura Muzzo
5273,72f764c1,Emma Stevanin
5274,a7a8fc6e,Vittoria Zanette


In [12]:
players[players["player_name"].apply(lambda name: "Halaholo" in name)]

,player_id,player_name
931,c18c9ebc,Clem Halaholo


In [22]:
import os
dirpath = "notebooks/rugbydata/player"
fs = pd.concat([pd.read_parquet(os.path.join(dirpath, f), engine="pyarrow") for f in os.listdir(dirpath) if "player_registry" not in f])

In [53]:
sel_match = pd.read_parquet("notebooks/rugbydata/match/f9896230.parquet", engine="pyarrow")
sel_match.columns

Index(['match_id', 'date', 'season', 'competition_id', 'competition',
       'venue_id', 'venue', 'city_played', 'home_team', 'away_team',
       'home_team_id', 'away_team_id', 'completed', 'is_tournament',
       'played_on_grass', 'attendance', 'home_team_form', 'away_team_form',
       'kickoff_time', 'home_score', 'away_score', 'referee'],
      dtype='str')

In [36]:
fs.columns

Index(['player_id', 'name', 'team', 'team_id', 'position', 'carries',
       'line_breaks', 'tackles_completed', 'turnovers_lost', 'turnovers_won',
       'kicks', 'passes', 'metres_carried', 'offloads', 'defenders_beaten',
       'try_assists', 'tries', 'tackles_made', 'tackles_missed',
       'dominant_tackles', 'ruck_turnovers', 'lineouts_won',
       'carries_per_minute', 'total_tackles_per_minute', 'red_cards',
       'yellow_cards', 'penalties_conceded', 'game_date', 'competition_id',
       'competition', 'match_id', 'team_vs', 'team_vs_id', 'weight_kg',
       'height_cm', 'clean_breaks', 'conversion_goals', 'drop_goals_converted',
       'kick_percent_success', 'kicks_from_hand', 'lineout_won_steal',
       'mauls_won', 'meters_run', 'missed_tackles', 'offload', 'penalty_goals',
       'points', 'rucks_won', 'runs', 'tackles', 'total_free_kicks_conceded',
       'total_lineouts', 'turnover_knock_on', 'turnovers_conceded'],
      dtype='str')

In [47]:
import math
bad_match_ids = fs[fs.apply(lambda p: not math.isnan(p["runs"]) and not math.isnan(p["carries"]) and p["runs"] != p["carries"], axis=1)]["match_id"]

In [48]:
bad_matches = pd.concat([pd.read_parquet(f"notebooks/rugbydata/match/{name}.parquet", engine="pyarrow") for name in bad_match_ids.unique()])

In [49]:
bad_matches

,match_id,date,season,competition_id,competition,venue_id,venue,city_played,home_team,away_team,...,completed,is_tournament,played_on_grass,attendance,home_team_form,away_team_form,kickoff_time,home_score,away_score,referee
0,75f5c4a4,20231020,2023,2c3df351,Gallagher Prem,74f2b7f9,Kingston Park,Newcastle,Newcastle Falcons,Gloucester Rugby,...,True,True,True,None,WLLWW,LLLLW,16:00,14,18,None
0,e83a4699,20231029,2023,2c3df351,Gallagher Prem,74f2b7f9,Kingston Park,Newcastle,Newcastle Falcons,Northampton Saints,...,True,True,True,None,WLLWW,WWWWL,16:00,14,16,None
0,6c149cbe,20220904,2022,814f16bc,Mitre 10 Cup,a10f0ba7,North Harbour Stadium,Albany,North Harbour,Counties Manukau,...,True,True,True,None,WLLLL,LWWWW,16:00,37,12,None
0,9bf2959b,20220910,2022,814f16bc,Mitre 10 Cup,a10f0ba7,North Harbour Stadium,Albany,North Harbour,Bay of Plenty,...,True,True,True,None,WLLLL,LWWWW,16:00,34,48,None
0,378d860a,20220813,2022,814f16bc,Mitre 10 Cup,da851e91,Apollo Projects Stadium,Christchurch,Canterbury,Wellington,...,True,True,True,None,WWWWT,LLWLL,16:00,43,10,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
0,efea22cc,20231014,2023,2c3df351,Gallagher Prem,95766442,Recreation Ground,Bath,Bath Rugby,Newcastle Falcons,...,True,True,True,None,LWLWW,WLLWW,16:00,34,26,None
0,0156edcf,20240511,2024,2c3df351,Gallagher Prem,a038d7a1,Ashton Gate,Bristol,Bristol Rugby,Saracens,...,True,True,True,None,WWWWW,WLLWL,16:00,20,41,None
0,f164f168,20231028,2023,2c3df351,Gallagher Prem,81e9ced3,Sandy Park,Exeter,Exeter Chiefs,Sale Sharks,...,True,True,True,None,WWTWW,LWLWL,16:00,43,0,None
0,f461114d,20240518,2024,50e6935b,United Rugby Championship,7fc4631c,Ellis Park,Johannesburg,Lions,Glasgow Warriors,...,True,True,True,None,LLLWW,WWWWL,16:00,44,21,None


In [27]:
date = "20231021"
fs.query("game_date == @date")

,player_id,name,team,team_id,position,carries,line_breaks,tackles_completed,turnovers_lost,turnovers_won,...,offload,penalty_goals,points,rucks_won,runs,tackles,total_free_kicks_conceded,total_lineouts,turnover_knock_on,turnovers_conceded
53,006ba58e,Javan Sebastian,Edinburgh,242b5b2e,prop,0.0,0.0,4.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,4.0,0.0,0.0,0.0,0.0
37,00e279b7,Andrea Zambonin,Zebre,8ab9f1c9,lock,1.0,0.0,8.0,1.0,0.0,...,0.0,0.0,0.0,0.0,1.0,8.0,0.0,0.0,1.0,1.0
0,00e3c4fc,Jordan Edmunds,Edinburgh,242b5b2e,default,0.0,0.0,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
50,0112444e,Curwin Bosch,Sharks,1b2d3d61,fly_half,5.0,1.0,6.0,1.0,0.0,...,0.0,0.0,6.0,2.0,5.0,6.0,0.0,0.0,0.0,1.0
10,012e836e,Tom Dodd,Edinburgh,242b5b2e,flanker,10.0,0.0,9.0,2.0,1.0,...,1.0,0.0,0.0,8.0,10.0,9.0,0.0,0.0,1.0,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
45,fcf24d6f,Owen Lane,Cardiff Blues,79c81be4,centre,6.0,0.0,1.0,0.0,0.0,...,1.0,0.0,0.0,5.0,6.0,1.0,0.0,0.0,0.0,0.0
46,fd68486e,Seb Davies,Cardiff Blues,79c81be4,lock,10.0,0.0,14.0,1.0,0.0,...,0.0,0.0,0.0,10.0,10.0,14.0,0.0,0.0,1.0,1.0
31,fe453724,Tom Seabrook,Northampton Saints,895ae027,centre,9.0,2.0,5.0,2.0,0.0,...,1.0,0.0,10.0,6.0,9.0,5.0,0.0,0.0,2.0,2.0
42,ff37baee,Caolin Blade,Connacht,6f5af01b,scrum_half,6.0,2.0,3.0,3.0,0.0,...,0.0,0.0,15.0,1.0,6.0,3.0,0.0,0.0,2.0,3.0
